# RK1828 YOLO Object Detection Demo (YOLOv5/v6/v8 + YOLO26)

Run 6 YOLO models in one go and understand the two output formats:

| Model | Output | Post-processing |
|---|---|---|
| YOLOv5 / v6 / v8 | `[1,K,6]` (score, class, x1, y1, x2, y2) | decode + NMS baked into the model; only coordinate restore needed |
| YOLO26 (n/s/m) | 6 raw heads (box + score at strides 8/16/32) | host-side sigmoid + box decode + per-class NMS |

| Step | Content |
|---|---|
| 1 | Download models (from huggingface.co) |
| 2 | Pick a model and an image |
| 3 | Pre-processing (letterbox: aspect-preserving resize + gray pad) |
| 4 | RKNN inference |
| 5 | Post-processing and visualization |

## 0. Environment check

Make sure the RKNN3 runtime (`librknn3_api.so`) and the RK1828 device are ready.
If this fails, install the driver/runtime first (the `rknn3-rk182x` DEB package)
and check with `sudo rknn-smi info` that the RK1828 shows up.

In [ ]:
from rknn3 import get_lib, find_devices

_ = get_lib()  # loads /usr/lib/librknn3_api.so; failure means the runtime is not installed

devices = find_devices()
if not devices:
    raise RuntimeError("No RK1828 device found! Check: sudo rknn-smi info")
for dev_id, dev_type in devices:
    print(f"Found device: id={dev_id}  interface={dev_type}")

## 1. Download models

The models for this demo are hosted on Hugging Face (account `qhddz`) and are
downloaded directly from **huggingface.co**. Run the next cell to start.

> Files that already exist (and are non-empty) are **skipped automatically** —
> re-running never downloads them twice.
> If your network cannot reach huggingface.co, set up a proxy first, e.g.
> `export https_proxy=http://<your-proxy>:<port>` before starting Jupyter.


This demo downloads YOLOv5/v6/v8 and YOLO26 n/s/m (i8 and fp16 each), plus the COCO 80-class labels and sample images.

In [ ]:
# (HF repo, path in repo, local path)
FILES = [
    ("yolo-rknn3-rk1828", "yolov5/i8/yolov5.rknn", "models/yolo/yolov5/i8/yolov5.rknn"),
    ("yolo-rknn3-rk1828", "yolov5/i8/yolov5.weight", "models/yolo/yolov5/i8/yolov5.weight"),
    ("yolo-rknn3-rk1828", "yolov5/fp16/yolov5.rknn", "models/yolo/yolov5/fp16/yolov5.rknn"),
    ("yolo-rknn3-rk1828", "yolov5/fp16/yolov5.weight", "models/yolo/yolov5/fp16/yolov5.weight"),
    ("yolo-rknn3-rk1828", "yolov6/i8/yolov6.rknn", "models/yolo/yolov6/i8/yolov6.rknn"),
    ("yolo-rknn3-rk1828", "yolov6/i8/yolov6.weight", "models/yolo/yolov6/i8/yolov6.weight"),
    ("yolo-rknn3-rk1828", "yolov6/fp16/yolov6.rknn", "models/yolo/yolov6/fp16/yolov6.rknn"),
    ("yolo-rknn3-rk1828", "yolov6/fp16/yolov6.weight", "models/yolo/yolov6/fp16/yolov6.weight"),
    ("yolo-rknn3-rk1828", "yolov8/i8/yolov8.rknn", "models/yolo/yolov8/i8/yolov8.rknn"),
    ("yolo-rknn3-rk1828", "yolov8/i8/yolov8.weight", "models/yolo/yolov8/i8/yolov8.weight"),
    ("yolo-rknn3-rk1828", "yolov8/fp16/yolov8.rknn", "models/yolo/yolov8/fp16/yolov8.rknn"),
    ("yolo-rknn3-rk1828", "yolov8/fp16/yolov8.weight", "models/yolo/yolov8/fp16/yolov8.weight"),
    ("yolo-rknn3-rk1828", "yolo26n/i8/yolo26n.rknn", "models/yolo/yolo26n/i8/yolo26n.rknn"),
    ("yolo-rknn3-rk1828", "yolo26n/i8/yolo26n.weight", "models/yolo/yolo26n/i8/yolo26n.weight"),
    ("yolo-rknn3-rk1828", "yolo26n/fp16/yolo26n.rknn", "models/yolo/yolo26n/fp16/yolo26n.rknn"),
    ("yolo-rknn3-rk1828", "yolo26n/fp16/yolo26n.weight", "models/yolo/yolo26n/fp16/yolo26n.weight"),
    ("yolo-rknn3-rk1828", "yolo26s/i8/yolo26s.rknn", "models/yolo/yolo26s/i8/yolo26s.rknn"),
    ("yolo-rknn3-rk1828", "yolo26s/i8/yolo26s.weight", "models/yolo/yolo26s/i8/yolo26s.weight"),
    ("yolo-rknn3-rk1828", "yolo26s/fp16/yolo26s.rknn", "models/yolo/yolo26s/fp16/yolo26s.rknn"),
    ("yolo-rknn3-rk1828", "yolo26s/fp16/yolo26s.weight", "models/yolo/yolo26s/fp16/yolo26s.weight"),
    ("yolo-rknn3-rk1828", "yolo26m/i8/yolo26m.rknn", "models/yolo/yolo26m/i8/yolo26m.rknn"),
    ("yolo-rknn3-rk1828", "yolo26m/i8/yolo26m.weight", "models/yolo/yolo26m/i8/yolo26m.weight"),
    ("yolo-rknn3-rk1828", "yolo26m/fp16/yolo26m.rknn", "models/yolo/yolo26m/fp16/yolo26m.rknn"),
    ("yolo-rknn3-rk1828", "yolo26m/fp16/yolo26m.weight", "models/yolo/yolo26m/fp16/yolo26m.weight"),
    ("yolo26-segment-rknn3-rk1828", "model/coco_80_labels_list.txt", "datasets/coco_80_labels_list.txt"),
    ("yolo26-segment-rknn3-rk1828", "model/bus.jpg", "datasets/bus.jpg"),
    ("mobilenet_v2-rknn3-rk1828", "dog_224x224.jpg", "datasets/dog_224x224.jpg"),
]

In [ ]:
from nb_download import ensure_files

ensure_files(FILES, "hf")

## 2. Pick a model and an image

Model files live under `models/yolo/<name>/<quant>/`; feel free to use your own image.

**Actual variant of each model** (judged from weight sizes):

| Directory | Actual variant | Params | Note |
|---|---|---|---|
| `yolov5` | **YOLOv5s** | 7.2M | i8/fp16 directories match their content |
| `yolov6` | **YOLOv6s** | 17.2M | ⚠️ `i8`/`fp16` directories **appear swapped** (the "i8" dir has the larger, slower weights — measured) |
| `yolov8` | **YOLOv8s** | 11.2M | ⚠️ same as above, `i8`/`fp16` **appear swapped** |
| `yolo26n/s/m` | YOLO26n / 26s / 26m | 2.4M / 9.5M / 20.4M | directories match their content |

> The v6/v8 directory issue comes from the official Rockchip SDK conversion
> package itself (the i8 build embeds an fp16 NMS post-processing head, making
> it bigger and slower); trust the measurements, not the directory names.

In [ ]:
import glob
import ipywidgets as widgets
from IPython.display import display

w_model = widgets.Dropdown(
    options=["yolov5", "yolov6", "yolov8", "yolo26n", "yolo26s", "yolo26m"],
    value="yolov8", description="Model:")
w_quant = widgets.Dropdown(options=["i8", "fp16"], description="Quant:")
w_image = widgets.Dropdown(options=sorted(glob.glob("datasets/*.jpg")),
                           description="Image:", layout=widgets.Layout(width="60%"))
display(w_model, w_quant, w_image)

LABELS = open("datasets/coco_80_labels_list.txt").read().splitlines()
print(f"{len(LABELS)} COCO classes. Then keep running downwards ▼")

## 3. Pre-processing: letterbox

YOLO expects a 640x640 square input. Stretching would distort aspect ratio, so
we **resize proportionally and pad with gray**, remembering the scale and
padding — the post-processing needs them to map boxes back to original
coordinates.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from vision_post import letterbox, to_input_array, setup_cjk_font

setup_cjk_font()
img = Image.open(w_image.value).convert("RGB")
img_lb, SCALE, PAD_X, PAD_Y = letterbox(img, 640)
x = to_input_array(img_lb)
print(f"original {img.size} -> input {x.shape}, scale={SCALE:.3f}, pad=({PAD_X},{PAD_Y})")

# show every pre-processing stage: original -> letterbox -> input tensor
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, im, title in [
    (axes[0], img, f"1. original {img.size[0]}x{img.size[1]}"),
    (axes[1], img_lb, "2. letterbox 640x640\n(aspect resize + gray pad)"),
    (axes[2], x[0], f"3. model input {x.shape}\nuint8 NHWC"),
]:
    ax.imshow(np.asarray(im))
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. RKNN inference

In [ ]:
import time
from rknn3 import Rknn3Model

model_path = f"models/yolo/{w_model.value}/{w_quant.value}/{w_model.value}.rknn"
model = Rknn3Model(model_path)
print(f"{len(model.output_attrs)} outputs:")
for a in model.output_attrs:
    print("  ", a)

_ = model.infer([x])
t0 = time.time()
outputs = model.infer([x])
print(f"inference time: {(time.time()-t0)*1000:.1f} ms")

## 5. Post-processing

- **YOLOv5/v6/v8**: output is already post-NMS `(score, class, x1, y1, x2, y2)`;
  just filter by confidence
- **YOLO26**: 6 heads (3 box heads `[1,4,h,w]` + 3 score heads `[1,80,h,w]`);
  sigmoid, box decode and per-class NMS run on the host; i8 outputs are
  dequantized first

In [ ]:
from vision_post import (decode_builtin_nms, decode_yolo26_heads,
                         unletterbox_boxes)

if w_model.value.startswith("yolo26"):
    boxes, classes, scores = decode_yolo26_heads(outputs, model.output_attrs,
                                                 conf_thresh=0.4)
else:
    boxes, classes, scores = decode_builtin_nms(outputs[0], conf_thresh=0.4)

boxes = unletterbox_boxes(boxes, SCALE, PAD_X, PAD_Y, *img.size)
print(f"{len(boxes)} objects detected:")
for b, c, s in zip(boxes, classes, scores):
    print(f"  {LABELS[c]:12s} {s:.2f}  box=({b[0]:.0f},{b[1]:.0f},{b[2]:.0f},{b[3]:.0f})")

## 6. Visualization

In [ ]:
from vision_post import draw_boxes

result = draw_boxes(img, boxes, classes, scores, LABELS)
model.close()
result

## 7. Multi-model comparison (i8 + fp16, full sweep)

Same image, running **6 models x 2 quantizations (i8 / fp16) = 12 combos**.
Progress is printed live; afterwards you get a comparison table, a latency
bar chart, and a detection mosaic. Each model is timed 3 times and the best
is kept (warm-up jitter excluded).

In [ ]:
import time
from rknn3 import Rknn3Model
from vision_post import (decode_builtin_nms, decode_yolo26_heads,
                         unletterbox_boxes)

# directory -> actual variant: yolov5=YOLOv5s, yolov6=YOLOv6s, yolov8=YOLOv8s
# note: yolov6/yolov8 i8 and fp16 directories appear swapped (official SDK
# package, see the table above)
ALL_MODELS = ["yolov5", "yolov6", "yolov8", "yolo26n", "yolo26s", "yolo26m"]
QUANTS = ["i8", "fp16"]

compare = []  # (model, quant, latency_ms, boxes, classes, scores)
total = len(ALL_MODELS) * len(QUANTS)
step = 0
for name in ALL_MODELS:
    for quant in QUANTS:
        step += 1
        print(f"[{step:2d}/{total}] loading {name} ({quant}) ...", flush=True)
        m = Rknn3Model(f"models/yolo/{name}/{quant}/{name}.rknn")
        m.infer([x])  # warmup
        best, outs = float("inf"), None
        for _ in range(3):
            t0 = time.time()
            outs = m.infer([x])
            best = min(best, (time.time() - t0) * 1000)
        if name.startswith("yolo26"):
            b, c, s = decode_yolo26_heads(outs, m.output_attrs, conf_thresh=0.4)
        else:
            b, c, s = decode_builtin_nms(outs[0], conf_thresh=0.4)
        b = unletterbox_boxes(b, SCALE, PAD_X, PAD_Y, *img.size)
        compare.append((name, quant, best, b, c, s))
        m.close()
        print(f"      -> {best:6.1f} ms, {len(b)} objects", flush=True)
print("\nAll done ✅", flush=True)

### Comparison table

In [ ]:
from IPython.display import HTML

rows = "".join(
    f"<tr><td>{n}</td><td>{q}</td>"
    f"<td>{lat:.1f}</td><td>{len(b)}</td>"
    f"<td>{', '.join(f'{LABELS[ci]}({si:.2f})' for ci, si in list(zip(c, s))[:5]) or '-'}</td></tr>"
    for n, q, lat, b, c, s in compare)
HTML("<table style='text-align:left'>"
     "<tr><th>Model</th><th>Quant</th><th>Latency (ms)</th><th>Objects</th><th>Top-5 detections</th></tr>"
     + rows + "</table>")

### Latency bar chart (i8 vs fp16)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from vision_post import setup_cjk_font

setup_cjk_font()
colors = {"i8": "#3b82f6", "fp16": "#f59e0b"}
xs = np.arange(len(ALL_MODELS))
width = 0.38
fig, ax = plt.subplots(figsize=(10, 4.5))
for k, q in enumerate(QUANTS):
    lats = [lat for n, qq, lat, *_ in compare if qq == q]
    cnts = [len(b) for n, qq, _, b, *_ in compare if qq == q]
    bars = ax.bar(xs + (k - 0.5) * width, lats, width, label=q, color=colors[q])
    for bar, lat, cnt in zip(bars, lats, cnts):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                f"{lat:.0f}ms\n{cnt} obj", ha="center", va="bottom", fontsize=8)
ax.set_xticks(xs, ALL_MODELS)
ax.set_ylabel("inference latency (ms)")
ax.set_title("YOLO multi-model comparison (same image, i8 vs fp16)")
ax.set_ylim(0, max(lat for _, _, lat, *_ in compare) * 1.3)
ax.legend()
plt.tight_layout()
plt.show()

### Detection mosaic (top row i8, bottom row fp16)

In [ ]:
import matplotlib.pyplot as plt
from vision_post import draw_boxes

fig, axes = plt.subplots(2, 6, figsize=(20, 7))
for k, q in enumerate(QUANTS):
    row = [(n, lat, b, c, s) for n, qq, lat, b, c, s in compare if qq == q]
    for ax, (n, lat, b, c, s) in zip(axes[k], row):
        ax.imshow(draw_boxes(img, b, c, s, LABELS))
        ax.set_title(f"{n} ({q})  {lat:.0f}ms  {len(b)} obj", fontsize=10)
        ax.axis("off")
plt.tight_layout()
plt.show()

## Summary

- YOLOv5/v6/v8 bake decode + NMS into the model, so the host does almost no
  post-processing; YOLO26 emits raw heads for deployment flexibility, with
  post-processing on the host (see `vision_post.decode_yolo26_heads`).
- General rule: i8 beats fp16; bigger models are slower but detect more;
  official reference mAP@0.5:0.95: 26n 40.9 / 26s 48.6 / 26m 53.1 (PyTorch FP32).
- Variants: v5/v6/v8 are the official Rockchip SDK pre-converted models,
  released by family name. Judging by weight size they are all **s variants**:
  `yolov5`=YOLOv5s (7.2M params), `yolov6`=YOLOv6s (17.2M),
  `yolov8`=YOLOv8s (11.2M) — **none of them is an n variant**.
  Also the v6/v8 `i8`/`fp16` directories appear swapped (the "i8" dir has the
  larger, slower weights), because the i8 build embeds an fp16 NMS head;
  trust measurements when picking a quantization.